<a href="https://www.kaggle.com/code/avikdas567/digitization-of-ecg-images?scriptVersionId=291484041" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports & Config

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# Paths

In [2]:
DATA_ROOT = "/kaggle/input/physionet-ecg-image-digitization"
TEST_CSV = os.path.join(DATA_ROOT, "test.csv")
SAMPLE_SUB = os.path.join(DATA_ROOT, "sample_submission.parquet")
TEST_IMG_DIR = os.path.join(DATA_ROOT, "test")

# Load Metadata

In [3]:
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_parquet(SAMPLE_SUB)

print("Test rows:", len(test_df))
print("Submission rows:", len(sample_sub))

Test rows: 24
Submission rows: 75000


# Utility: Expected Length

In [4]:
def expected_length(fs, lead):
    if lead == "II":
        return int(np.floor(fs * 10.0))
    else:
        return int(np.floor(fs * 2.5))

# Image Preprocessing

In [5]:
def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    return gray

# Waveform Extraction

In [6]:
def extract_waveform(gray_img, target_len):
    """
    Extract waveform by column-wise minimum intensity
    """
    h, w = gray_img.shape

    # Normalize width to expected length
    x_positions = np.linspace(0, w - 1, target_len).astype(int)

    waveform = []

    for x in x_positions:
        col = gray_img[:, x]
        y = np.argmin(col)   # darkest pixel → ECG ink
        waveform.append(y)

    waveform = np.array(waveform, dtype=np.float32)

    # Invert y-axis and normalize
    waveform = -waveform
    waveform -= waveform.mean()
    waveform /= (np.std(waveform) + 1e-6)
    waveform *= 0.5  # scale to reasonable mV range

    return waveform

# Generate Predictions

In [7]:
records = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    base_id = row["id"]
    lead = row["lead"]
    fs = row["fs"]

    img_path = os.path.join(TEST_IMG_DIR, f"{base_id}.png")
    img = cv2.imread(img_path)

    if img is None:
        # fallback safety
        length = expected_length(fs, lead)
        signal = np.zeros(length)
    else:
        gray = preprocess_image(img)
        length = expected_length(fs, lead)
        signal = extract_waveform(gray, length)

    for i, val in enumerate(signal):
        records.append({
            "id": f"{base_id}_{i}_{lead}",
            "value": float(val)
        })

100%|██████████| 24/24 [00:02<00:00, 10.27it/s]


# Build Submission

In [8]:
submission = pd.DataFrame(records)

submission = submission.set_index("id").loc[sample_sub["id"]].reset_index()

submission.head()

,id,value
0,1053922973_0_I,0.836637
1,1053922973_1_I,0.836637
2,1053922973_2_I,0.836637
3,1053922973_3_I,0.836637
4,1053922973_4_I,0.836637


# Save Submission

In [9]:
submission.to_csv("submission.csv", index=False)

print("Saved submission.csv")
print("Rows:", len(submission))

Saved submission.csv
Rows: 75000
